# 《PythAPCS123》單元 13-5：語意錯誤（Logic Error）與常見邏輯盲點排查（WA 防範）

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者  
**學習目標**：
- 深刻理解語意錯誤（Logic Error）的本質，剖析線上評判系統「答案錯誤（WA, Wrong Answer）」的成因
- 地毯式排查 APCS 考場高頻 7 大邏輯盲點：差一錯誤（Off-by-one）、運算優先級、浮點精度與負數除法、二維陣列淺拷貝幽靈、變數遮蔽污染、多測資狀態殘留、極端邊界（Corner Cases）漏判
- 掌握競賽選手必備神技「對拍檢測（Stress Testing）」：以隨機生成器與雙標程自動揪出最小出錯反例
- 建立考場送出前邏輯自我檢驗 SOP，徹底告別「自以為寫對卻拿 0 分」的 WA 悲劇


### 13.5.1 什麼是語意錯誤（Logic Error）？語法完全合法但評判慘遭 WA 的本質

在程式設計的世界中，最令人挫折的不是看到滿螢幕紅字的語法錯誤（SyntaxError）或執行時期崩潰（RuntimeError），而是**程式碼語法百分之百正確、順利執行至第 100 行結束且中途沒有拋出任何例外，但最後印出來的答案卻與題目要求完全不符**。這就是所謂的**語意錯誤（Semantic Error / Logic Error，邏輯錯誤）**。

在電腦科學界有一句名言：「電腦永遠只會精確執行你寫出來的指令，而不會執行你心中所想像的意圖（The computer did exactly what you told it to do, not what you wanted it to do.）。」
- 語法錯誤（CE）是直譯器看不懂你的文法；
- 執行時期錯誤（RE）是直譯器碰上了無法執行的物理運算（如除零、越界）；
- 而語意錯誤則是**直譯器完完全全聽懂了你的話，並且忠實執行到底，但你的思考邏輯一開始就偏離了航道**！

在 APCS 考場與線上評判系統（Online Judge）中，當你的程式順利執行完成，評判伺服器會將你程式的標準輸出（stdout）逐字逐行與出題老師預先準備的「官方標準答案」進行字元級的比對。只要任何一個測試案例的數值、正負號、筆數或排序有哪怕一丁點的偏差，系統就會毫不留情地給出刺眼的 **`WA (Wrong Answer，答案錯誤)`** 評判狀態！

特別危險的是：初學者往往只會拿題目下方附帶的 1 到 2 個簡單「公開範例測資」來驗證程式。當看到公開範例輸出吻合時，便滿懷自信地按下送出；殊不知命題老師在系統後台準備了大量涵蓋各種極端邊界與特殊條件的「隱藏測資」。一旦思考盲點被隱藏測資精準命中，整題就直接宣告淪陷。因此，學會剖析邏輯漏洞並學會反思極端條件，是邁向 APCS 實作滿分的核心基石！


In [ ]:
# 13.5.1 程式碼演示：語意錯誤現場 —— 語法完全正確但運算結果背離題意
# 題目要求：給定整數 n，計算 1 到 n 之間所有偶數的平方和（包含 n）
# 正確數學公式：若 n=6，偶數為 2, 4, 6，平方和為 4 + 16 + 36 = 56

def sum_even_squares_buggy(n):
    # 語法百分之百合法！直譯器不會報錯，執行也不會崩潰！
    # 但內部藏有致命的語意邏輯錯誤：
    # 漏洞 1: range(n) 只跑到 n-1，遺漏了剛好為偶數的 n 本身！
    # 漏洞 2: 忘記取平方，直接累加偶數數值！
    total = 0
    for i in range(n):
        if i % 2 == 0:
            total += i # 邏輯錯誤：累加了 i 而非 i 的平方
    return total

def sum_even_squares_correct(n):
    # 修正後的正確邏輯
    total = 0
    for i in range(1, n + 1):
        if i % 2 == 0:
            total += i * i
    return total

n_test = 6
buggy_ans = sum_even_squares_buggy(n_test)
correct_ans = sum_even_squares_correct(n_test)

print(f"輸入測試值 n = {n_test}")
print(f"考生程式輸出（零報錯順利跑完）: {buggy_ans}")
print(f"官方標準答案（正確邏輯預期）: {correct_ans}")

if buggy_ans != correct_ans:
    print("\n❌ 評判系統判定: [WA (Wrong Answer)]！")
    print(f"差距分析: 考生輸出 {buggy_ans} != 正確標準 {correct_ans}，一分未得！")


### 13.5.1 語法重點回顧與核心觀念提煉

1. **WA 的本質是邏輯偏差而非程式語言錯誤**：電腦完全照著你的指示執行，但你的演算法與題意要求存在落差。
2. **範例 AC 不等於全面 AC**：範例測資通常偏小、偏理想；隱藏測資才是真正檢驗邏輯完備性的試金石。
3. **除錯首要任務：量化差異**：遭遇 WA 時，第一件事是手動用筆紙手算一個極小案例，對比程式每一行中間變數的真實數值與預期數值！


In [ ]:
# 13.5.1 學生實作練習：輸出比對與 WA 診斷器
# 任務說明：實作 diagnose_wa_discrepancy(expected_output, actual_output)
# 接收兩個多行文字字串 expected_output 與 actual_output
# 逐行去除行末空白後比對：
# 1. 若兩者完全相同，回傳 ("AC", "Accepted")
# 2. 若行數不同或內容不符，找到「第一個出現差異的行號（1-based）」並回傳：
#    ("WA", f"Line {line_no}: expected '{exp}', got '{act}'")

def diagnose_wa_discrepancy(expected_output: str, actual_output: str):
    exp_lines = [l.rstrip() for l in expected_output.strip().split("\n")]
    act_lines = [l.rstrip() for l in actual_output.strip().split("\n")]
    
    max_lines = max(len(exp_lines), len(act_lines))
    for i in range(max_lines):
        exp = exp_lines[i] if i < len(exp_lines) else "<EOF>"
        act = act_lines[i] if i < len(act_lines) else "<EOF>"
        if exp != act:
            return ("WA", f"Line {i + 1}: expected '{exp}', got '{act}'")
            
    return ("AC", "Accepted")

# 測試用例
print("測試完全相符:", diagnose_wa_discrepancy("10\n20", "10\n20 "))
print("測試數值差異:", diagnose_wa_discrepancy("56", "6"))
print("測試行數不足:", diagnose_wa_discrepancy("1\n2\n3", "1\n2"))


In [ ]:
# 13.5.1 單元測試驗證
assert diagnose_wa_discrepancy("42", "42") == ("AC", "Accepted")
assert diagnose_wa_discrepancy("Line1\nLine2", "Line1\nLine2 ") == ("AC", "Accepted")
assert diagnose_wa_discrepancy("100", "99") == ("WA", "Line 1: expected '100', got '99'")
assert diagnose_wa_discrepancy("A\nB\nC", "A\nB") == ("WA", "Line 3: expected 'C', got '<EOF>'")
print("🎉 13.5.1 所有測試通過！成功建立 WA 評判本質與診斷意識！")


### 13.5.2 邏輯盲點一：差一錯誤（Off-by-one Error，開閉區間與次數偏差）

在程式設計歷史上，**「差一錯誤（Off-by-one Error）」**被公認為最經典、最高頻、且最難一眼看穿的隱形殺手。它的特點是程式架構完全正確，但迴圈次數偏偏「多跑了一次」或「少跑了一次」，或者在區間計算時長度「多算了 1」或「少算了 1」。

考場中最容易引爆差一錯誤的四大情境：
1. **`range()` 的「左閉右開」特性誤用**：
   - Python 的 `range(start, stop)` 是**包含 start，但不包含 stop（即 $[start, stop)$）**！
   - 題目要求：「計算 $1$ 到 $n$ 的累加和」。初學者極常手滑寫出 `range(1, n)`，這會導致當 $i = n$ 時迴圈已經提前結束，永遠少加了最後一個數字 $n$！正確寫法必為 `range(1, n + 1)`。
2. **閉區間 $[L, R]$ 元素個數計算偏差**：
   - 題目給定一段編號從 $L$ 到 $R$ 的連續號碼，求總共有幾個人？
   - 人類的幾何直覺常下意識寫成 $R - L$（例如編號 3 到 5 號，直覺 $5 - 3 = 2$ 個人）。但實際上是 3 號、4 號、5 號共 3 個人！閉區間元素個數公式永遠是：
     $$\text{Count} = R - L + 1$$
3. **二分搜尋或雙指標相遇條件 `<` vs `<=` 偏差**：
   - 雙指標夾擊時，終止條件到底該寫 `while left < right:` 還是 `while left <= right:`？
   - 如果兩指標相遇時（`left == right`）的那一格仍有意義（例如單元搜尋需檢驗最後一個元素），寫 `<` 就會漏掉中間正中央的關鍵數字，直接吃下 WA！
4. **字串或串列切片邊界偏移**：
   - 取字串前 $k$ 個字元：`s[0:k]`（長度為 $k$）；若手滑寫成 `s[0:k-1]` 則只取了 $k-1$ 個字元。

🛡️ **防禦心法：代入極小值檢驗法（Extreme Value Check）**
- 當你不確定閉區間要不要加 1、或是迴圈要不要寫 `n + 1` 時，立刻在腦中代入極端測試案例：
  - 假設區間只有 1 個數：$L = 3, R = 3$。正確個數應該是 1。
  - 公式代入：$R - L = 0$（錯！）；$R - L + 1 = 1$（正確！驗算完畢，立刻安心加上 `+ 1`）。


In [ ]:
# 13.5.2 程式碼演示：差一錯誤三重災難現場與極小值驗算心法
print("--- 案例 1: range(1, n) 遺漏末項慘劇 ---")
n = 5
# 考生錯誤寫法：
buggy_sum = sum(range(1, n)) # 實際上只加了 1 + 2 + 3 + 4 = 10
# 正確寫法：
correct_sum = sum(range(1, n + 1)) # 1 + 2 + 3 + 4 + 5 = 15

print(f"求 1~{n} 總和: 考生輸出 {buggy_sum} vs 正確答案 {correct_sum} (差一漏掉 {n}！)")

print("\n--- 案例 2: 閉區間 [L, R] 人數計算偏差 ---")
L, R = 3, 7
wrong_count = R - L
correct_count = R - L + 1
print(f"區間 [{L}, {R}] 包含元素個數:")
print(f"  直覺運算 R - L = {wrong_count} (少算 1 個！)")
print(f"  公式驗證 R - L + 1 = {correct_count} (正確答案: {list(range(L, R + 1))})")

print("\n--- 案例 3: 雙指標相遇條件漏判單一元素 ---")
def find_target_buggy(arr, target):
    left, right = 0, len(arr) - 1
    # 錯誤條件：使用 < 導致當只剩最後一個元素時跳出，未進行比對！
    while left < right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return True
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return False

test_arr = [42]
print(f"在陣列 {test_arr} 中搜尋 42:")
print(f"  考生寫 while left < right 結果: {find_target_buggy(test_arr, 42)} (慘遭 WA！)")


### 13.5.2 語法重點回顧與核心觀念提煉

1. **閉區間牢記加一**：
   - 題目說明「包含頭尾」或「$1$ 到 $n$」：迴圈右邊界必寫 `n + 1`！
   - 閉區間 $[L, R]$ 元素總數：必寫 `R - L + 1`！
2. **極小值代入驗算法**：
   - 拿 $n=1$ 或 $L=R$ 驗算，0 秒破解是否該加 1 的猶豫不決。
3. **雙指標邊界檢查**：
   - 閉區間搜尋標準模板永遠使用 `while left <= right:`。


In [ ]:
# 13.5.2 學生實作練習：閉區間長度與安全累加器
# 任務說明：實作兩個精準防禦差一錯誤的函式
# 1. interval_stats(L, R)：
#    給定整數 L, R（保證 L <= R），計算閉區間 [L, R] 內：
#    (1) 整數個數 count
#    (2) 所有整數的總和 total_sum
#    回傳元組 (count, total_sum)。要求精確無誤，嚴禁出現差一錯誤！
# 2. safe_binary_search(arr, target)：
#    在已排序串列 arr 中搜尋 target 是否存在。存在回傳 True，否則回傳 False。
#    要求：長度為 1 的陣列必須能被正確檢索，不得遺漏！

def interval_stats(L: int, R: int) -> tuple:
    # 請在此處實作精準閉區間統計
    count = R - L + 1
    # 善用等差數列公式：(首項 + 末項) * 項數 // 2
    total_sum = (L + R) * count // 2
    return (count, total_sum)

def safe_binary_search(arr: list, target: int) -> bool:
    # 請在此處使用正確的邊界條件實作二分搜尋
    left, right = 0, len(arr) - 1
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return True
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return False

# 測試用例
print("區間 [3, 5] 統計:", interval_stats(3, 5))
print("單點區間 [7, 7] 統計:", interval_stats(7, 7))
print("單元素陣列二分搜尋:", safe_binary_search([10], 10))
print("單元素陣列找不到:", safe_binary_search([10], 5))


In [ ]:
# 13.5.2 單元測試驗證
assert interval_stats(1, 5) == (5, 15)
assert interval_stats(10, 10) == (1, 10)
assert interval_stats(-3, 3) == (7, 0)

assert safe_binary_search([5], 5) == True
assert safe_binary_search([5], 6) == False
assert safe_binary_search([], 1) == False
assert safe_binary_search([1, 3, 5, 7, 9], 9) == True
print("🎉 13.5.2 所有測試通過！徹底消滅差一錯誤（Off-by-one）隱形殺手！")


### 13.5.3 邏輯盲點二：運算優先級與結合性陷阱（算式偏差與 not/and/or/位元混淆）

在高中數學中，我們都知道「先乘除、後加減」。在程式語言中，運算子的數量遠遠超過數學課本，涵蓋了算術、比較、邏輯、位元等多達數十種運算子。當多個運算子混雜在同一行運算式時，如果不加小括號輔助，直譯器會依照內建的**優先級規則（Operator Precedence）**進行結合，這常常導致運算順序與學生的預期南轅北轍，引發隱蔽的 WA！

考場最高頻引發 WA 的三大運算優先級天坑：
1. **計算兩數平均值漏括號**：
   - 考場最經典手滑算式：`avg = a + b // 2`。
   - 因為整除 `//` 的優先級高於加號 `+`，這行算式實際上被電腦理解為：$a + \frac{b}{2}$，而不是 $\frac{a + b}{2}$！
   - 正確寫法必為 `(a + b) // 2`。
2. **邏輯運算子優先級：`not > and > or`**：
   - 初學者直覺常以為邏輯運算子是「由左至右平鋪直敘」。
   - 例如條件式：`True or False and False`。直譯器執行的順序是 `True or (False and False)` $\rightarrow$ 結果為 `True`！
   - 但初學者的直覺通常想表達的是「先判斷前面成立與否，再與後面相與」，即 `(True or False) and False` $\rightarrow$ 結果應為 `False`！兩者結論完全相反，造成條件判斷嚴重錯亂！
3. **競賽經典至尊大坑：位移運算優先級低於加減法（`1 << n - 1` 陷阱）**：
   - 在狀態壓縮與位元運算中，生成「長度為 $n$ 的全 1 遮罩（Mask）」公式為 $2^n - 1$。
   - 許多考生順手寫出：`mask = 1 << n - 1`。
   - ⚠️ **死神降臨**：在 Python 運算子優先級中，**減法 `-` 的優先級高於位移 `<<`**！
   - 因此這行式子會被直譯器解析為：`1 << (n - 1)`！以 $n = 3$ 為例，$2^3 - 1 = 7$（二進位 `111`）；但 `1 << 3 - 1` 算出來卻是 `1 << 2 = 4`（二進位 `100`），遮罩完全失真，整題爆零！
   - 正確寫法：**位移運算必須明確加上小括號**：`(1 << n) - 1`！

🛡️ **防禦心法：括號不嫌多，意圖更明確！**
在考場上，千萬不要盲目炫技去背誦複雜的優先級表。只要運算式涉及混合運算子（尤其是算術混位移、邏輯 not/and/or 混用），**毫不猶豫加上小括號 `()`**！小括號的優先級永遠最高，還能大幅提升程式碼的可讀性。


In [ ]:
# 13.5.3 程式碼演示：三大優先級天坑現場實況
print("--- 天坑 1: 平均值計算漏括號 ---")
a, b = 10, 20
wrong_avg = a + b // 2     # 被解析為 10 + (20 // 2) = 20
correct_avg = (a + b) // 2 # 正確 (10 + 20) // 2 = 15
print(f"求 {a} 與 {b} 的平均值:")
print(f"  漏括號 a + b // 2 = {wrong_avg} (算錯！)")
print(f"  加括號 (a + b) // 2 = {correct_avg} (正確！)")

print("\n--- 天坑 2: 邏輯 and 優先於 or 陷阱 ---")
# 題目意圖：(A or B) 成立的前提下，且 C 必須成立
A, B, C = True, False, False
wrong_logic = A or B and C      # 被解析為 True or (False and False) -> True
correct_logic = (A or B) and C  # 正確 (True or False) and False -> False
print("邏輯運算 A or B and C:")
print(f"  漏括號 A or B and C = {wrong_logic} (邏輯顛倒！)")
print(f"  加括號 (A or B) and C = {correct_logic} (正確！)")

print("\n--- 天坑 3: 競賽位元遮罩 1 << n - 1 陷阱 ---")
# 意圖生成 n=3 的全 1 遮罩（即 2^3 - 1 = 7，二進位 111）
n = 3
wrong_mask = 1 << n - 1      # 減法搶先：1 << (3 - 1) = 1 << 2 = 4 (二進位 100)
correct_mask = (1 << n) - 1  # (1 << 3) - 1 = 8 - 1 = 7 (二進位 111)

print(f"生成長度為 {n} 的全 1 遮罩 (預期 7):")
print(f"  漏括號 1 << n - 1 = {wrong_mask} (算成 4，嚴重 WA！)")
print(f"  加括號 (1 << n) - 1 = {correct_mask} (正確！)")


### 13.5.3 語法重點回顧與核心觀念提煉

1. **算術優先級口訣**：
   - 括號第一、乘除第二、加減第三。只要分子是多項式相加，分子必加小括號！
2. **位元運算鐵律**：
   - `&`, `|`, `^`, `<<`, `>>` 遇到比較運算子（`==`, `!=`, `<`, `>`），**一律用小括號把位元運算包起來**！
3. **邏輯否定括起來**：
   - 想要否定整個複合條件時，必須寫 `not (A and B)`，絕不可省略括號。


In [ ]:
# 13.5.3 學生實作練習：安全優先級計算器
# 任務說明：實作兩個精準括號運算函式
# 1. safe_midpoint(low, high)：
#    計算整數 low 與 high 的中點 (low + high) // 2，杜絕漏括號偏差！
# 2. is_even_bitwise(n)：
#    使用位元運算 & 判斷整數 n 是否為偶數。若是偶數回傳 True，奇數回傳 False。
#    要求：必須正確加上括號防範 == 優先級搶先結合問題！

def safe_midpoint(low: int, high: int) -> int:
    # 請在此處實作正確括號之中點計算
    return (low + high) // 2

def is_even_bitwise(n: int) -> bool:
    # 請在此處使用位元運算 & 搭配正確括號判斷偶數
    return (n & 1) == 0

# 測試用例
print("中點計算 (10, 20):", safe_midpoint(10, 20))
print("中點計算 (3, 7):", safe_midpoint(3, 7))
print("判斷 4 是否為偶數:", is_even_bitwise(4))
print("判斷 7 是否為偶數:", is_even_bitwise(7))
print("判斷 0 是否為偶數:", is_even_bitwise(0))


In [ ]:
# 13.5.3 單元測試驗證
assert safe_midpoint(10, 20) == 15
assert safe_midpoint(0, 8) == 4
assert safe_midpoint(-10, 10) == 0

assert is_even_bitwise(4) == True
assert is_even_bitwise(7) == False
assert is_even_bitwise(0) == True
assert is_even_bitwise(-2) == True
assert is_even_bitwise(-3) == False
print("🎉 13.5.3 所有測試通過！徹底打破運算子優先級混淆陷阱！")


### 13.5.4 邏輯盲點三：浮點數精確度與除法語意陷阱

在計算機底層硬體中，資料是以二進位（Binary，基底為 2）儲存的。二進位浮點數（IEEE 754 標準）在表示十進位的小數時，就像十進位無法精確表示 $\frac{1}{3} = 0.33333...$ 一樣，在二進位中甚至連 $0.1$ 和 $0.2$ 都無法被精準整除，只能儲存為一個極度接近但帶有微小截斷誤差的無窮循環二進位小數。

考場四大浮點數與除法邏輯盲點：
1. **直接比對浮點數相等性（`==` 陷阱）**：
   - 在 Python 終端機執行 `0.1 + 0.2 == 0.3`，答案竟然是令人震驚的 **`False`**！
   - 因為 `0.1 + 0.2` 的真實內部數值是 `0.30000000000000004`。若你在條件判斷中寫 `if total == 0.3:`，這道題就直接吃下 WA！
   - **正確比對心法**：浮點數絕不可用 `==`，必須改用「容許極小誤差範圍（Epsilon, $\epsilon$）」：
     `if abs(a - b) < 1e-9:`！
2. **整數除法 `//` 在負數上的「向下取整（Floor Division）」陷阱**：
   - 初學者從 C/C++ 或 Java 轉來時，最容易在此翻車！
   - 在 C/C++ 中，整數除法是「向零截斷（Truncate towards Zero）」：$-7 / 2 = -3$。
   - 但在 Python 中，`//` 嚴格定義為「向下取整（Floor，向負無窮大方向取整）」：
     $$-7 // 2 = -4$$
   - 同理，取餘數模除：Python 的 `-7 % 2 = 1`（在 C++ 中為 `-1`）。如果題目是移植自 C/C++ 題目且包含負數座標位移，未注意此差異將導致答案整列偏離！
3. **Python 內建 `round()` 的「銀行家捨入法（Round to Even）」陷阱**：
   - 考題若要求「四捨五入取至整數」，學生直覺呼叫 `round(2.5)`，預期得到 3；但 Python 的 `round(2.5)` 實際上回傳 **`2`**！
   - Python 的 `round()` 採用統計學上的銀行家捨入法：當剛好處於 .5 時，會捨入到**最接近的偶數**（`round(2.5) == 2`, `round(3.5) == 4`）。這常與國中小數學嚴格的「四捨五入」標準答案產生衝突。

🛡️ **防禦心法**：
- 浮點相等用 `abs(x - y) < 1e-9`。
- 純粹四捨五入至整數：利用 `int(x + 0.5)`（正數情況）或標準庫 `decimal` 模組！


In [ ]:
# 13.5.4 程式碼演示：浮點數精度丟失、負數整除偏差與銀行家捨入法
print("--- 陷阱 1: 0.1 + 0.2 != 0.3 驚魂記 ---")
sum_float = 0.1 + 0.2
print(f"0.1 + 0.2 的實際底層數值: {sum_float:.20f}")
print(f"直接使用 == 比對結果: {sum_float == 0.3} (慘遭 False！)")

# 安全比對做法：
def is_equal_float(a, b, eps=1e-9):
    return abs(a - b) < eps

print(f"🛡️ 使用 Epsilon 容許誤差比對: {is_equal_float(sum_float, 0.3)} (成功判定相等！)")

print("\n--- 陷阱 2: 負數整數除法向負無窮取整 ---")
print(f" 7 // 2 正數整除: {7 // 2}")
print(f"-7 // 2 負數整除（Python 向下取整為 -4，而非 -3！）: {-7 // 2}")
print(f"-7 % 2 負數模除: {-7 % 2}")

print("\n--- 陷阱 3: 內建 round() 銀行家捨入法偏差 ---")
# 銀行家捨入法：.5 捨入到最接近的偶數
print(f"round(2.5) -> {round(2.5)} (不是 3！因為 2 是偶數)")
print(f"round(3.5) -> {round(3.5)} (4 是偶數)")

# 考場標準數學「四捨五入（正數）」手刻公式：
def round_half_up_positive(x):
    return int(x + 0.5)

print(f"🛡️ 嚴格四捨五入 round_half_up_positive(2.5): {round_half_up_positive(2.5)} (成功得到 3！)")


### 13.5.4 語法重點回顧與核心觀念提煉

1. **浮點比較唯一守則**：
   - 嚴格禁止 `float_a == float_b`！
   - 一律寫成 `abs(float_a - float_b) < 1e-9`。
2. **負數向零截斷技巧**：
   - 若題目要求 C 語言風格的向零截斷：寫成 `int(a / b)` 而非 `a // b`！
3. **正數四捨五入必備小工具**：
   - `int(x + 0.5)` 能百分之百保證 .5 必定進位到下一個整數。


In [ ]:
# 13.5.4 學生實作練習：浮點精度守門員與向零截斷除法器
# 任務說明：實作兩個精準數值處理函式
# 1. safe_float_equal(a, b, tolerance=1e-9)：
#    判斷浮點數 a 與 b 是否在容許誤差範圍內相等。若相等回傳 True，否則回傳 False。
# 2. truncate_towards_zero_divide(a, b)：
#    實作「向零截斷」整數除法（模擬 C/C++ 行為）。
#    例如 7 與 2 回傳 3；-7 與 2 回傳 -3（而非 Python 預設的 -4）！

def safe_float_equal(a: float, b: float, tolerance: float = 1e-9) -> bool:
    # 請在此處使用 abs 實作精度容許比對
    return abs(a - b) < tolerance

def truncate_towards_zero_divide(a: int, b: int) -> int:
    # 請在此處使用 int(a / b) 實作向零截斷
    return int(a / b)

# 測試用例
print("比對 0.1+0.2 是否等於 0.3:", safe_float_equal(0.1 + 0.2, 0.3))
print("向零截斷正數 7 // 2:", truncate_towards_zero_divide(7, 2))
print("向零截斷負數 -7 // 2:", truncate_towards_zero_divide(-7, 2))


In [ ]:
# 13.5.4 單元測試驗證
assert safe_float_equal(0.1 + 0.2, 0.3) == True
assert safe_float_equal(1.00000000001, 1.0) == True
assert safe_float_equal(1.05, 1.0) == False

assert truncate_towards_zero_divide(7, 2) == 3
assert truncate_towards_zero_divide(-7, 2) == -3
assert truncate_towards_zero_divide(10, -3) == -3
assert truncate_towards_zero_divide(-10, -3) == 3
print("🎉 13.5.4 所有測試通過！成功駕馭浮點數精度與除法語意！")


### 13.5.5 邏輯盲點四：淺拷貝（Shallow Copy）與物件共享連動幽靈

在 APCS 實作第二題與第三題中，二維地圖走訪、矩陣旋轉與棋盤模擬是長年不衰的經典題型。然而，幾乎每一屆都有大量初學者在宣告二維地圖的第一步就掉入了萬劫不復的深淵——**`grid = [[0] * m] * n` 連動修改幽靈**！

什麼是二維陣列淺拷貝連動幽靈？
- 當初學者想要宣告一個 $3 \times 3$ 的全零矩陣時，常寫出：`grid = [[0] * 3] * 3`。
- **記憶體底層黑幕揭秘**：
  內層的 `[0] * 3` 確實建立了一個包含三個 0 的串列物件（假設在記憶體位址 `0x100`）。但外層的 `* 3`，並**沒有**去複製建立三個獨立的串列，而是**將同一個記憶體位址 `0x100` 的參照（Reference）重複貼了 3 次**！
- 產生的恐怖災難：
  當你想把第 0 列第 0 行改為 1，執行 `grid[0][0] = 1` 時，因為三列全部共用同一個記憶體空間，**第 1 列和第 2 列的第 0 行也會同步詭異地變成 1**！整個地圖被神秘連動污染，考生盯著螢幕百思不得其解，最終慘吞 WA。

此外，還有兩種常見的物件共用陷阱：
1. **簡單等號賦值別名（Alias）**：
   `list_b = list_a`。這並不是複製串列，而是幫同一個串列取了別名。修改 `list_b` 會直接改毀 `list_a`！要複製必須寫 `list_b = list_a.copy()` 或 `list_b = list_a[:]`。
2. **函式可變預設參數陷阱（Mutable Default Argument）**：
   `def add_item(x, lst=[]): lst.append(x); return lst`。在 Python 中，預設參數只會在函式定義編譯時建立一次，多次呼叫會共用同一個串列，造成跨呼叫污染！

🛡️ **防禦心法：二維陣列唯一正解宣告公式**
在 Python 中建立二維陣列，唯一安全且標準的寫法是**串列生成式（List Comprehension）**：
`grid = [[0] * m for _ in range(n)]`！
每一次迴圈迭代都會真真切切在記憶體中建立一個全新的獨立串列，徹底剷除共享連動幽靈！


In [ ]:
# 13.5.5 程式碼演示：二維陣列連動污染現場與生成式安全防禦
print("--- 恐怖幽靈現場: [[0] * 3] * 3 記憶體共用災難 ---")
buggy_grid = [[0] * 3] * 3
print("初始矩陣狀態:")
for row in buggy_grid:
    print(" ", row)

print("\n執行指令: buggy_grid[0][0] = 99 (只改左上角一格)")
buggy_grid[0][0] = 99

print("修改後矩陣狀態（三列全部被連動污染！）:")
for idx, row in enumerate(buggy_grid):
    print(f"  列 {idx} (id={id(row)}): {row}")

print("\n--- 🛡️ 安全防禦模式: 串列生成式 List Comprehension ---")
safe_grid = [[0] * 3 for _ in range(3)]
print("執行指令: safe_grid[0][0] = 99")
safe_grid[0][0] = 99

print("安全矩陣狀態（僅左上角被修改，每列擁有獨立記憶體位址）:")
for idx, row in enumerate(safe_grid):
    print(f"  列 {idx} (id={id(row)}): {row}")


### 13.5.5 語法重點回顧與核心觀念提煉

1. **二維矩陣宣告唯一鐵律**：
   - 建立 $R$ 列 $C$ 行：`grid = [[0] * C for _ in range(R)]`。
   - 嚴格禁止外層使用 `* R`！
2. **串列拷貝記得切片**：
   - 想要複製一維串列：`b = a[:]` 或 `b = a.copy()`。
   - 想要複製二維結構：使用 `import copy; b = copy.deepcopy(a)`。
3. **預設參數避坑**：
   - 函式參數切勿設定為 `lst=[]`，應使用 `lst=None` 並在函式內部初始化。


In [ ]:
# 13.5.5 學生實作練習：獨立二維矩陣建構器與無副作用串列操作
# 任務說明：實作兩個物件防污染函式
# 1. create_independent_matrix(rows, cols, initial_val=0)：
#    建立一個 rows x cols 的二維矩陣，每個元素初始為 initial_val。
#    要求：修改任一格的值，絕不可影響其他列的數值！
# 2. safe_append_copy(original_list, new_element)：
#    傳入串列 original_list 與新元素 new_element。
#    回傳一個加入該元素的新串列，要求不得修改或破壞原始 original_list 的內容！

def create_independent_matrix(rows: int, cols: int, initial_val = 0) -> list:
    # 請在此處使用串列生成式實作獨立矩陣建立
    return [[initial_val] * cols for _ in range(rows)]

def safe_append_copy(original_list: list, new_element) -> list:
    # 請在此處實作純函式無副作用拷貝加入
    new_lst = original_list.copy()
    new_lst.append(new_element)
    return new_lst

# 測試用例
grid = create_independent_matrix(2, 3, 0)
grid[0][1] = 77
print("獨立矩陣驗證 列0:", grid[0])
print("獨立矩陣驗證 列1 (應保持全0):", grid[1])

orig = [1, 2, 3]
copied = safe_append_copy(orig, 99)
print("原始清單:", orig)
print("新產出清單:", copied)


In [ ]:
# 13.5.5 單元測試驗證
m = create_independent_matrix(3, 4, 0)
m[1][2] = 5
assert m[0] == [0, 0, 0, 0]
assert m[1] == [0, 0, 5, 0]
assert m[2] == [0, 0, 0, 0]

src = ["a", "b"]
res = safe_append_copy(src, "c")
assert res == ["a", "b", "c"]
assert src == ["a", "b"] # 原始串列未受污染
print("🎉 13.5.5 所有測試通過！徹底粉碎二維陣列淺拷貝共享幽靈！")


### 13.5.6 邏輯盲點五：變數遮蔽與全域/內建名稱污染

在撰寫較長或具有多重巢狀結構的程式時，如果變數命名的規劃混亂，極容易發生**變數遮蔽（Variable Shadowing）**。當內層範圍的變數名稱與外層變數（或 Python 內建名稱）完全相同時，外層變數就會被無情遮蔽覆蓋，導致程式計算完全失真，引發詭異難解的 WA！

考場中最嚴重的三大變數遮蔽翻車現場：
1. **巢狀迴圈計數器同名「互相踩腳」**：
   - 考生在寫雙層迴圈走訪矩陣時，外層寫了 `for i in range(rows):`。
   - 寫到內層時打字太順手，竟然又寫了 `for i in range(cols):`！
   - 當內層迴圈跑完時，變數 `i` 的數值已經被內層修改得面目全非，外層的計數控制直接被破壞瓦解，迴圈提前終止或重複執行，整張矩陣的運算徹底報銷！
2. **遮蔽 Python 內建關鍵函式**：
   - 初學者極常宣告：`sum = 0`、`max = 10`、`min = 5` 或 `list = []`。
   - 當這行執行後，全域命名空間的內建函式 `sum()` 就被你的整數變數取代！後續程式如果想要呼叫 `total = sum([1, 2, 3])`，就會直接被直譯器丟出 `TypeError`，或者在判斷式中引發數值混亂！
3. **函式內部參數遮蔽全域配置**：
   - 在主程式定義了全域變數 `target = 100`，但在輔助函式定義時參數也命名為 `def check(target):`。函式內部所存取的 `target` 將完全與全域變數脫鉤，若本意是參照全域狀態，就會產生巨大的邏輯偏差。

🛡️ **防禦心法：命名加後綴，維度明確化**：
- 雙層迴圈維度清晰化：外層用 `r`（row）或 `i`，內層務必用 `c`（col）或 `j`！三層迴圈使用 `k`。
- 數值累計命名規範：總和使用 `total_sum`、最大值使用 `max_val`、清單使用 `num_list`，永遠嚴禁單獨使用 `sum`, `max`, `min`, `list`, `dict`, `set`, `str`, `int` 作為變數名稱！


In [ ]:
# 13.5.6 程式碼演示：巢狀迴圈同名變數互相踩腳與遮蔽災難
print("--- 案例 1: 巢狀迴圈 i 遮蔽外層 i 災難現場 ---")
matrix = [
    [1, 2],
    [3, 4]
]

def traverse_buggy(mat):
    print("  [錯誤演示] 外層與內層都用 i:")
    log = []
    for i in range(len(mat)): # 外層 i = 0, 1
        for i in range(len(mat[0])): # 內層又把 i 覆蓋成 0, 1！
            log.append(f"mat[{i}][{i}]={mat[i][i]}")
    return log

print("  踩腳執行軌跡:", traverse_buggy(matrix))

def traverse_safe(mat):
    print("\n  [正確演示] 外層用 r，內層用 c:")
    log = []
    for r in range(len(mat)):
        for c in range(len(mat[0])):
            log.append(f"mat[{r}][{c}]={mat[r][c]}")
    return log

print("  標準執行軌跡:", traverse_safe(matrix))


### 13.5.6 語法重點回顧與核心觀念提煉

1. **二維索引命名慣例**：
   - 行列坐標：`r` 代表列（Row），`c` 代表行（Column）。
   - 笛卡爾坐標：`x` 代表橫軸，`y` 代表縱軸。
   - 走訪變數絕對不重名，杜絕互相覆蓋。
2. **內建名稱保護紅線**：
   - 看到編輯器中變數名稱變成了特殊顏色（高亮語法），表示它碰到了 Python 關鍵字或內建名稱，立刻加上 `_val` 或 `_sum`！


In [ ]:
# 13.5.6 學生實作練習：安全矩陣對角線計算器
# 任務說明：實作 matrix_diagonal_sums(mat)
# 給定一個 N x N 的方形二維矩陣 mat（保證行數等於列數）
# 計算並回傳一個元組 (main_diag_sum, anti_diag_sum)：
# 1. 主對角線總和（左上到右下，座標 [i][i]）
# 2. 次對角線總和（右上到左下，座標 [i][N - 1 - i]）
# 要求：嚴禁使用 sum 作為變數名，嚴禁索引變數遮蔽衝突！

def matrix_diagonal_sums(mat: list) -> tuple:
    n = len(mat)
    main_sum = 0
    anti_sum = 0
    # 請在此處使用安全不遮蔽的變數名稱實作對角線加總
    for i in range(n):
        main_sum += mat[i][i]
        anti_sum += mat[i][n - 1 - i]
    return (main_sum, anti_sum)

# 測試用例
test_square = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
]
# 主對角線: 1 + 5 + 9 = 15; 次對角線: 3 + 5 + 7 = 15
print("3x3 矩陣對角線總和:", matrix_diagonal_sums(test_square))


In [ ]:
# 13.5.6 單元測試驗證
m1 = [
    [1, 2],
    [3, 4]
]
assert matrix_diagonal_sums(m1) == (5, 5) # (1+4, 2+3)

m2 = [
    [2, 0, 0],
    [0, 3, 0],
    [0, 0, 4]
]
assert matrix_diagonal_sums(m2) == (9, 3) # (2+3+4, 0+3+0)
print("🎉 13.5.6 所有測試通過！徹底杜絕變數遮蔽與名稱污染！")


### 13.5.7 邏輯盲點六：多測資狀態殘留與迴圈計數器未歸零

在 APCS 考場與 ZeroJudge 系統上，常常會看到考生發出這樣的哀嚎：「**明明在本地終端機手動輸入第 1 筆測資答案完全正確，為什麼一送上去線上評判直接全盤吃下 WA？**」

這就是考場中最具欺騙性的隱形幽靈——**「多測資狀態殘留（Multi-testcase State Leakage）」**！
許多 APCS 實作題目採用「單一測試檔包含多筆獨立測資」的輸入架構（例如以 EOF 結尾、或是開頭先給定測資組數 $T$）。每一筆測資之間在邏輯上應該是「完全獨立、毫不相干」的平行世界。

然而，許多初學者的變數宣告習慣存在致命弱點：
- **致命弱點：將累加變數、集合標記宣告在多測資迴圈的「外部」**！
- 運作慘狀剖析：
  1. 程式讀取第 1 筆測資，累加器從 0 開始加到 15，輸出 15（答案正確！）。
  2. 程式接著進入第 2 筆測資，本應重新從 0 開始累加；但因為累加器宣告在外層迴圈上方，它**保留了第 1 筆測資殘留下來的 15**！第 2 筆測資算出來的答案直接多了 15，當場被評判系統判定 WA！
  3. 到了第 3 筆測資，數值如同雪球般越滾越大，全盤皆墨。

同樣的慘劇也頻繁發生在**布林旗標（Flag）與已造訪集合（`visited`）**上：
- 檢查條件旗標 `found = False`，在第 1 筆測資命中目標後變為 `True`；在進入第 2 筆測資時忘記重設回 `False`，導致第 2 筆測資一開始就被誤判為已經找到！

🛡️ **防禦心法：最小作用域原則（Principle of Least Privilege）**
- **變數在哪個範圍使用，就在哪個範圍的最開頭初始化**！
- 處理多測資的 while 或 for 迴圈內部第 1 行，就是重設所有累加器、計數器與容器清單的神聖陣地！絕不在外層殘留任何全域狀態。


In [ ]:
# 13.5.7 程式碼演示：多測資狀態殘留慘劇模擬與最小作用域重構
print("=== 模擬多測資輸入情境 ===")
# 兩筆獨立測資：每筆測資包含一組數字，求各組的正數個數
competition_cases = [
    [10, -5, 20],  # 案例 1: 有 2 個正數 (10, 20)
    [-1, -2, 30]   # 案例 2: 有 1 個正數 (30)
]

def solve_multicase_buggy(all_cases):
    print("  [錯誤寫法] 計數器宣告在迴圈外面:")
    # 致命手滑：count 宣告在外面！
    positive_count = 0
    results = []
    
    for case_idx, arr in enumerate(all_cases, 1):
        for x in arr:
            if x > 0:
                positive_count += 1
        results.append(positive_count)
        print(f"    第 {case_idx} 筆測資計算結果: {positive_count}")
    return results

buggy_outputs = solve_multicase_buggy(competition_cases)
print(f"  --> 考生輸出: {buggy_outputs} (案例 2 殘留案例 1 的結果，變成 3 了！WA！)")

def solve_multicase_correct(all_cases):
    print("\n  [正確寫法] 進入每筆測資立即重設狀態:")
    results = []
    
    for case_idx, arr in enumerate(all_cases, 1):
        # 🛡️ 在每筆測資開頭重新歸零初始化
        positive_count = 0
        for x in arr:
            if x > 0:
                positive_count += 1
        results.append(positive_count)
        print(f"    第 {case_idx} 筆測資安全計算結果: {positive_count}")
    return results

correct_outputs = solve_multicase_correct(competition_cases)
print(f"  --> 🛡️ 正確輸出: {correct_outputs} (每筆獨立精準計算！)")


### 13.5.7 語法重點回顧與核心觀念提煉

1. **多測資重設口訣**：
   - 「一進迴圈立刻歸零，歷史包袱不留痕跡。」
2. **三項必清空狀態清單**：
   - 數值累加器（`total = 0`, `count = 0`）
   - 布林狀態旗標（`found = False`, `is_valid = True`）
   - 走訪記憶容器（`visited = set()`, `path = []`）


In [ ]:
# 13.5.7 學生實作練習：多組測資乾淨狀態統計器
# 任務說明：實作 process_testcases_cleanly(batch_data)
# batch_data 為二維串列，每個子串列代表一筆獨立的考試測資。
# 對於每一筆測資，請計算出「該測資中大於 50 的數值總和」。
# 回傳一個串列包含每筆測資的計算總和。
# 要求：每筆測資的加總必須完全獨立，嚴格禁止出現跨測資狀態污染！

def process_testcases_cleanly(batch_data: list) -> list:
    results = []
    # 請在此處實作乾淨無狀態污染之多測資處理
    for case in batch_data:
        sub_total = 0
        for val in case:
            if val > 50:
                sub_total += val
        results.append(sub_total)
    return results

# 測試用例
batches = [
    [60, 40, 70],  # 60 + 70 = 130
    [50, 51],      # 只有 51 大於 50 = 51
    [10, 20]       # 0
]
print("多測資批次乾淨統計結果:", process_testcases_cleanly(batches))


In [ ]:
# 13.5.7 單元測試驗證
assert process_testcases_cleanly([[60, 40, 70], [50, 51], [10, 20]]) == [130, 51, 0]
assert process_testcases_cleanly([[], [100]]) == [0, 100]
assert process_testcases_cleanly([[55], [55], [55]]) == [55, 55, 55]
print("🎉 13.5.7 所有測試通過！徹底拔除多測資狀態殘留隱形毒瘤！")


### 13.5.8 邏輯盲點七：極端邊界條件漏判（Corner Cases 遺漏）

在競技程式設計中，出題老師在撰寫測試資料產生器（Testcase Generator）時，心態往往是「處心積慮要揪出考生演算法中的漏洞」。那些位於數值範圍最邊緣、型態最極端、或是看似極不自然的特殊輸入，統稱為**邊界案例（Corner Cases / Edge Cases）**。

考場中奪走最多分數的四大極端 Corner Cases：
1. **最大值初始化設為 0 的「全負數」慘劇**：
   - 題目要求：「在輸入的一連串整數中，找出最大值」。
   - 初學者下意識寫出：`max_val = 0`。
   - ⚠️ **死穴**：如果輸入的測資全部都是負數（例如 `[-10, -5, -20]`），正確最大值應該是 `-5`。但因為你一開始把 `max_val` 設成 0，迴圈比對下來發現每個數都比 0 小，最後自信滿滿地輸出 `0`——而 `0` 根本連出現在輸入裡都沒有！直接吃下 WA！
   - **正確寫法**：
     - 方法 A：以負無窮大初始化 `max_val = -float('inf')`
     - 方法 B：直接拿輸入的第一個元素初始化 `max_val = nums[0]`！
2. **資料規模極小（$N=0$ 或 $N=1$）**：
   - 尋找第二大元素：如果測資只有 1 個數字，第二大元素根本不存在！
   - 陣列相鄰比對：如果長度為 1，相鄰探測根本無從發起，是否漏掉了特判？
3. **所有元素數值完全相同**：
   - 測資為 `[7, 7, 7, 7]`。嚴格大於 `>` 比對可能全部落空，導致輸出旗標未被觸發。
4. **題目規範的「無解特殊標記」漏判**：
   - 題目說明：「若無符合條件之解答，請輸出 `-1` 或 `IMPOSSIBLE`」。
   - 考生在主邏輯寫完了搜尋，卻忘記在找不到答案時輸出該特殊字串，導致輸出空行或預設 0，吞下全盤 WA。

🛡️ **防禦心法：極值初始化三原则**
- 求最大值：初始值設為「負無窮大 `-float('inf')`」或串列第 0 項。
- 求最小值：初始值設為「正無窮大 `float('inf')`」或串列第 0 項。
- 送出前自我質問：如果輸入全都是負數、如果 $N=1$、如果答案無解，我的程式會印出什麼？


In [ ]:
# 13.5.8 程式碼演示：全負數最大值初始化慘劇與第二大值邊界排查
print("--- 慘劇現場: max_val = 0 遭遇全負數測試資料 ---")
negative_inputs = [-15, -42, -8, -99]

def find_maximum_buggy(numbers):
    # 致命盲點：自以為 0 很小，預設為 0
    max_val = 0
    for x in numbers:
        if x > max_val:
            max_val = x
    return max_val

def find_maximum_correct(numbers):
    # 🛡️ 正確解法：初始化為負無窮大
    max_val = -float('inf')
    for x in numbers:
        if x > max_val:
            max_val = x
    return max_val

print(f"輸入測資: {negative_inputs}")
print(f"  考生寫 max_val = 0 輸出: {find_maximum_buggy(negative_inputs)} (WA！輸入明明沒 0！)")
print(f"  正確以負無窮大初始化輸出: {find_maximum_correct(negative_inputs)} (正確答案: -8！)")

print("\n--- 邊界排查: 尋找嚴格第二大值（無解特判）---")
def find_second_largest_safe(numbers):
    # 排除長度不足或全相同
    unique_nums = sorted(list(set(numbers)), reverse=True)
    if len(unique_nums) < 2:
        return None # 無解特判，絕不胡亂回傳垃圾值
    return unique_nums[1]

print("正常情況 [10, 20, 30] 第二大:", find_second_largest_safe([10, 20, 30]))
print("全相同情況 [5, 5, 5] 第二大:", find_second_largest_safe([5, 5, 5]))
print("單一元素 [42] 第二大:", find_second_largest_safe([42]))


### 13.5.8 語法重點回顧與核心觀念提煉

1. **極值初始化口訣**：
   - 「找最大設負無窮，找最小設正無窮；若有串列取首項，絕不憑空寫成零。」
2. **Corner Cases 必測四天王**：
   - 數值全負、數值全同、$N=1$、無解狀況。
3. **特判先行**：在函式最開頭用 `if len(arr) < 2:` 直接攔截極端邊界，讓主邏輯乾淨純粹。


In [ ]:
# 13.5.8 學生實作練習：抗全負數極值尋找與無解安全報告器
# 任務說明：實作 find_min_max_difference(numbers)
# 給定一個非空整數串列 numbers（可能包含全負數、重複數字或僅有 1 個數字）
# 計算並回傳最大值減最小值的差值 (max_val - min_val)。
# 要求：
# 1. 面對全負數測資必須能精準計算，嚴禁出現初始化為 0 造成的偏差！
# 2. 若 numbers 長度為 1，最大最小相同，差值應精確回傳 0。

def find_min_max_difference(numbers: list) -> int:
    # 請在此處實作安全的極值差值計算
    min_val = float('inf')
    max_val = -float('inf')
    for x in numbers:
        if x < min_val:
            min_val = x
        if x > max_val:
            max_val = x
    return max_val - min_val

# 測試用例
print("全負數測試 [-10, -5, -20]:", find_min_max_difference([-10, -5, -20])) # -5 - (-20) = 15
print("單一數字測試 [-42]:", find_min_max_difference([-42])) # 0
print("正負混合測試 [-5, 15, 0]:", find_min_max_difference([-5, 15, 0])) # 15 - (-5) = 20


In [ ]:
# 13.5.8 單元測試驗證
assert find_min_max_difference([-10, -5, -20]) == 15
assert find_min_max_difference([-100]) == 0
assert find_min_max_difference([5, 5, 5]) == 0
assert find_min_max_difference([-7, 8]) == 15
print("🎉 13.5.8 所有測試通過！徹底征服極端 Corner Cases 邊界盲點！")


### 13.5.9 考場 WA 極速診斷 SOP：對拍檢測（Stress Testing）與反例構造實戰

在 APCS 考場上，最絕望的時刻莫過於：「你寫完了一題看似很漂亮的演算法，送出後卻拿到刺眼的 **`WA`**；但線上評判系統為了公平性，**絕不可能公布它後台那組讓你出錯的隱藏測資長什麼樣子**！」此時如果只是漫無目的地在螢幕前空想，往往枯坐一小時也找不到錯誤。

競賽頂尖選手在考場面對 WA 時，有一套勝率高達 99% 的終極秘密武器——**「對拍檢測（Stress Testing，對拍除錯）」**！

什麼是對拍檢測？
1. **第一步：寫一個保證正確但速度慢的「暴力解（Brute Force / Slow）」**：
   - 考場通常要求 $O(N \log N)$ 或 $O(N)$，但寫一個最簡單無腦的雙層迴圈 $O(N^2)$ 暴力解通常只要 2 分鐘，且邏輯簡單到絕對不可能寫錯。
2. **第二步：寫一個隨機測資生成器（Random Generator）**：
   - 使用 Python 內建的 `random` 模組，隨機產生 100 組小規模的測資（例如陣列長度 $N \le 10$，數字大小在 $-20 \sim 20$ 之間）。
3. **第三步：啟動全自動對拍比對迴圈**：
   - 同時把隨機測資丟給「暴力保證解」與「你的優化解」；
   - 一旦兩者輸出的答案出現分歧，程式立刻緊急停止，並當場把這組**「讓你的優化程式翻車的最小反例（Counterexample）」**打印出來！
4. **第四步：手動單步追蹤最小反例**：
   - 拿著這組只有 3 到 5 個數字的精簡反例，在草稿紙上一步一步手算，1 分鐘內就能精確揪出到底是哪一行條件判斷寫錯！

這套「讓電腦自己找 bug」的方法，是從根源上徹底解決 WA 的終極技能！


In [ ]:
# 13.5.9 程式碼演示：考場自動「對拍除錯（Stress Test）」實戰示範
import random

# 題目：在串列中找出「兩數相乘的最大值」
# 考生優化寫法：以為只要找最大兩個正數相乘
def solve_fast_buggy(arr):
    # 致命盲點：完全忽略了「兩個極大負數相乘也會變成超大正數」！
    sorted_arr = sorted(arr)
    return sorted_arr[-1] * sorted_arr[-2]

# 暴力無腦保證解：雙層迴圈枚舉所有任兩數相乘，取最大值（絕對不會錯！）
def solve_slow_brute(arr):
    max_prod = -float('inf')
    n = len(arr)
    for i in range(n):
        for j in range(i + 1, n):
            prod = arr[i] * arr[j]
            if prod > max_prod:
                max_prod = prod
    return max_prod

print("=== 啟動對拍檢測迴圈（Stress Testing）===")
found_counterexample = False
random.seed(42) # 固定隨機種子便於重現

for iteration in range(1, 1001):
    # 隨機產生規模很小的測資（長度 4，數字 -10 ~ 10）
    size = random.randint(3, 6)
    test_case = [random.randint(-10, 10) for _ in range(size)]
    
    ans_brute = solve_slow_brute(test_case)
    ans_fast = solve_fast_buggy(test_case)
    
    if ans_brute != ans_fast:
        print(f"💥 抓到了！第 {iteration} 次對拍成功逮到出錯最小反例！")
        print(f"  出錯測試資料: {test_case}")
        print(f"  暴力保證解輸出: {ans_brute}")
        print(f"  考生優化解輸出: {ans_fast}")
        print("  💡 案情大白：該測資包含兩個極大負數（如 -10 * -9 = 90），優化解漏算了負負得正！")
        found_counterexample = True
        break

if not found_counterexample:
    print("全數通過對拍！")


### 13.5.9 語法重點回顧與核心觀念提煉

考場對拍四部曲：
1. **寫暴力**：不求效能，只求邏輯最直白、絕對正確的純暴力標程。
2. **造測資**：規模要小（方便人腦看懂），數值要包含 0 與負數。
3. **抓分歧**：比對兩者輸出，遇不同立刻 break 印出輸入。
4. **手追蹤**：拿著最小反例在草稿紙上還原案發現場，極速修正！


In [ ]:
# 13.5.9 學生實作練習：自動對拍測試驗證器
# 任務說明：實作 stress_test_validator(func_slow, func_fast, test_cases)
# 傳入暴力函式 func_slow、待檢驗函式 func_fast，以及多組測試案例清單 test_cases。
# 依序對每組測資比對兩函式的回傳值：
# 1. 若所有測資兩者結果完全一致，回傳 ("AC", len(test_cases))
# 2. 若在某組測資兩者回傳不同，立刻停止並回傳：
#    ("WA", failing_test_case, slow_ans, fast_ans)

def stress_test_validator(func_slow, func_fast, test_cases: list):
    # 請在此處實作對拍驗證邏輯
    for case in test_cases:
        slow_res = func_slow(case)
        fast_res = func_fast(case)
        if slow_res != fast_res:
            return ("WA", case, slow_res, fast_res)
    return ("AC", len(test_cases))

# 測試用例
def slow_square_sum(lst): return sum(x * x for x in lst)
def fast_square_sum_ok(lst): return sum(x ** 2 for x in lst)
def fast_square_sum_bad(lst): return sum(lst) # 故意寫錯

cases = [[1, 2], [3, 4], [0, 5]]
print("測試完全相符對拍:", stress_test_validator(slow_square_sum, fast_square_sum_ok, cases))
print("測試揪出錯誤對拍:", stress_test_validator(slow_square_sum, fast_square_sum_bad, cases))


In [ ]:
# 13.5.9 單元測試驗證
f_slow = lambda lst: max(lst) if lst else None
f_fast_good = lambda lst: sorted(lst)[-1] if lst else None
f_fast_bad = lambda lst: lst[0] if lst else None

assert stress_test_validator(f_slow, f_fast_good, [[1, 5, 2], [-5, -1], [10]]) == ("AC", 3)
res_wa = stress_test_validator(f_slow, f_fast_bad, [[10, 20]])
assert res_wa[0] == "WA"
assert res_wa[1] == [10, 20] # 抓出反例
assert res_wa[2] == 20
assert res_wa[3] == 10
print("🎉 13.5.9 所有測試通過！恭喜你已修成橫掃全場 WA 與對拍抓蟲的終極大師！")


## 13.5 總結與 WA 防禦地圖

在本單元中，我們針對線上評判系統最頑固、最磨人的 **語意錯誤（Logic Error / WA）** 進行了深入骨髓的系統化排查：

| 邏輯盲點類別 | 典型出錯現場 | 根治心法與防護公式 |
| :--- | :--- | :--- |
| **差一錯誤（Off-by-one）** | `range(1, n)` 漏末項、閉區間漏算 | 區間元素數必寫 $R - L + 1$；善用極小值代入驗算 |
| **運算優先級** | `a + b // 2` 算式偏差、`x & 1 == 0` 失效 | 算術分子加括號；位元運算周圍必加括號 `(x & 1) == 0` |
| **浮點與除法** | `0.1 + 0.2 != 0.3`、負數 `//` 向下取整 | 浮點比對用 `abs(a - b) < 1e-9`；向零截斷用 `int(a/b)` |
| **二維陣列淺拷貝** | `[[0] * m] * n` 連動污染整行變數 | 二維陣列唯一正解：`[[0] * m for _ in range(n)]` |
| **變數遮蔽污染** | 巢狀迴圈同用 `i`、變數命名 `sum = 0` | 外層用 `r` 內層用 `c`；命名加修飾詞（`total_sum`） |
| **多測資狀態殘留** | 累加器宣告在外層，第 2 筆繼承舊值 | 最小作用域原則：一進多測資迴圈立刻將變數歸零重設 |
| **Corner Cases 漏判** | 全負數時 `max_val = 0`、無解特判漏寫 | 求最大設負無窮大 `-float('inf')`；前置特判極端邊界 |
| **對拍除錯神技** | 遭遇 WA 卻看不到後台測試資料 | 寫暴力保證解 + 隨機小測資對拍，秒抓最小出錯反例 |

恭喜你攻克了最具挑戰性的語意錯誤大關！
在下一單元 **《13-6 時間超限（TLE）診斷：運算量估算、無窮迴圈與隱形效能坑洞防制》** 中，我們將直面另一大評判夢魘——超時（TLE），學習如何在考場估算 $10^7$ 運算極限並消除所有隱形效能殺手！
